In [15]:
import pandas as pd
import numpy as np
from pathlib import Path

In [16]:
BASE_PATH = Path("..")
RAW_PATH = BASE_PATH / "data" / "raw"
PROCESSED_PATH = BASE_PATH / "data" / "processed"

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

In [17]:
customers = pd.read_csv(RAW_PATH/'customers.csv')
orders = pd.read_csv(RAW_PATH/'orders.csv')
customer_features = pd.read_csv(RAW_PATH/'customer_features.csv')
customer_unit_economics = pd.read_csv(RAW_PATH/'customer_unit_economics.csv')
subscription_history = pd.read_csv(RAW_PATH/'subscription_history.csv')
marketing_spend = pd.read_csv(RAW_PATH/'marketing_spend.csv')
date_dim = pd.read_csv(RAW_PATH/'date_dim.csv')

In [18]:
## verify 

datasets ={
    'customers': customers,
    'orders': orders,
    'customer_features': customer_features,
    'customer_unit_economics': customer_unit_economics,
    'subscription_history': subscription_history,
    'marketing_spend': marketing_spend,
    'date_dim': date_dim
    
}


for name, df in datasets.items():
    print(f"{name}: {df.shape}")

customers: (204227, 11)
orders: (1126821, 15)
customer_features: (204227, 8)
customer_unit_economics: (204227, 10)
subscription_history: (2201060, 9)
marketing_spend: (960, 9)
date_dim: (1461, 7)


In [19]:
## Standardise column names

for name, df in datasets.items():
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )

In [20]:
for name, df in datasets.items():
    print(f"\n{name}")
    print(df.columns.tolist())


customers
['customer_id', 'signup_date', 'acquisition_channel', 'acquisition_month', 'acquisition_campaign_id', 'region', 'device_type', 'first_order_date', 'customer_status', 'date_of_birth', 'gender']

orders
['order_id', 'customer_id', 'order_date', 'product_category', 'quantity', 'revenue', 'discount', 'cost_of_goods', 'shipping_cost', 'delivery_region_cost', 'payment_processing_fee', 'order_status', 'refund_amount', 'net_revenue', 'gross_margin']

customer_features
['customer_id', 'order_frequency', 'avg_order_value', 'avg_discount_rate', 'days_since_last_order', 'avg_interpurchase_time', 'subscription_tenure', 'discount_dependency_ratio']

customer_unit_economics
['customer_id', 'total_revenue', 'total_discount', 'total_cogs', 'total_shipping', 'total_refunds', 'cac', 'contribution_margin', 'ltv', 'payback_months']

subscription_history
['subscription_id', 'customer_id', 'subscription_tier', 'start_date', 'end_date', 'monthly_fee', 'churn_flag', 'churn_reason', 'reactivation_fla

In [21]:
## Standardise data types

date_columns = {
    "customers": ["signup_date", "first_order_date", "date_of_birth"],
    "orders": ["order_date"],
    "subscription_history": ["start_date", "end_date"],
    "date_dim": ["date"]
}

for table, columns in date_columns.items():
    for col in columns:
        datasets[table][col] = pd.to_datetime(
            datasets[table][col],
            errors="coerce"
        )

marketing_spend["month"] = pd.to_datetime(
    marketing_spend["month"],
    format="%Y-%m",
    errors="coerce"
)

In [22]:
for name, df in datasets.items():
    date_cols = df.select_dtypes(include=["datetime"]).columns.tolist()
    if date_cols:
        print(f"{name}: {date_cols}")

customers: ['signup_date', 'first_order_date', 'date_of_birth']
orders: ['order_date']
subscription_history: ['start_date', 'end_date']
marketing_spend: ['month']
date_dim: ['date']


In [23]:
## Validate numerical columns

for name, df in datasets.items():
    print(f"\n--- {name} ---")
    print(df.select_dtypes(include=["number"]).dtypes)


--- customers ---
customer_id    int64
dtype: object

--- orders ---
order_id                    int64
customer_id                 int64
quantity                    int64
revenue                   float64
discount                  float64
cost_of_goods             float64
shipping_cost             float64
delivery_region_cost      float64
payment_processing_fee    float64
refund_amount             float64
net_revenue               float64
gross_margin              float64
dtype: object

--- customer_features ---
customer_id                    int64
order_frequency                int64
avg_order_value              float64
avg_discount_rate            float64
days_since_last_order          int64
avg_interpurchase_time       float64
subscription_tenure          float64
discount_dependency_ratio      int64
dtype: object

--- customer_unit_economics ---
customer_id              int64
total_revenue          float64
total_discount         float64
total_cogs             float64
total_shipping

In [24]:
## Handle missing values

missing_summary = pd.DataFrame({
    "missing_count": {
        name: df.isnull().sum().sum()
        for name, df in datasets.items()
    },
    "total_rows": {
        name: len(df)
        for name, df in datasets.items()
    }
})

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"] /
    missing_summary["total_rows"] * 100
)

missing_summary

,missing_count,total_rows,missing_percentage
customers,26376,204227,12.915041
orders,0,1126821,0.000000
customer_features,0,204227,0.000000
customer_unit_economics,0,204227,0.000000
subscription_history,2086112,2201060,94.777607
marketing_spend,0,960,0.000000
date_dim,0,1461,0.000000


In [25]:
## Verify missing-value patterns

print("Customers without first orders:")
print(
    customers["first_order_date"].isna().sum(),
    "out of",
    len(customers)
)

print("\nSubscriptions without churn reasons:")
print(
    subscription_history["churn_reason"].isna().sum(),
    "out of",
    len(subscription_history)
)

print("\nChurn flag distribution:")
print(subscription_history["churn_flag"].value_counts())

Customers without first orders:
26376 out of 204227

Subscriptions without churn reasons:
2086112 out of 2201060

Churn flag distribution:
churn_flag
0    2086112
1     114948
Name: count, dtype: int64


In [26]:
## Categorical data validation

categorical_columns = {
    "customers": [
        "acquisition_channel",
        "region",
        "device_type",
        "customer_status",
        "gender"
    ],
    "orders": [
        "product_category",
        "order_status"
    ],
    "subscription_history": [
        "subscription_tier",
        "churn_reason"
    ],
    "marketing_spend": [
        "channel",
        "region"
    ]
}

for table, columns in categorical_columns.items():
    print(f"\n========== {table} ==========")

    for col in columns:
        print(f"\n{col}:")
        print(datasets[table][col].value_counts(dropna=False).to_dict())


========== customers ==========

acquisition_channel:
{'Paid Ads': 81548, 'Organic': 57572, 'Referral': 34639, 'Influencer': 30468}

region:
{'South': 45107, 'North': 44882, 'West': 41042, 'East': 40695, 'Tier2': 32501}

device_type:
{'Mobile': 138764, 'Web': 65463}

customer_status:
{'Inactive': 103753, 'Active': 100474}

gender:
{'F': 106306, 'M': 89769, 'Other': 8152}

========== orders ==========

product_category:
{'Fashion': 281390, 'Beauty': 247690, 'Home': 225444, 'Electronics': 203024, 'Grocery': 169273}

order_status:
{'Completed': 980727, 'Returned': 100851, 'Cancelled': 45243}

========== subscription_history ==========

subscription_tier:
{'Basic': 1351577, 'Premium': 849483}

churn_reason:
{nan: 2086112, 'Inactivity': 39869, 'Price': 34728, 'Competitor': 23094, 'Other': 17257}

========== marketing_spend ==========

channel:
{'Paid Ads': 240, 'Organic': 240, 'Referral': 240, 'Influencer': 240}

region:
{'North': 192, 'South': 192, 'East': 192, 'West': 192, 'Tier2': 192}


In [27]:
## Check categorical consistency

print(
    "Customer channels:",
    set(customers["acquisition_channel"].unique())
)

print(
    "Marketing channels:",
    set(marketing_spend["channel"].unique())
)

print(
    "\nCustomer regions:",
    set(customers["region"].unique())
)

print(
    "Marketing regions:",
    set(marketing_spend["region"].unique())
)

Customer channels: {'Organic', 'Influencer', 'Paid Ads', 'Referral'}
Marketing channels: {'Organic', 'Referral', 'Influencer', 'Paid Ads'}

Customer regions: {'Tier2', 'South', 'East', 'North', 'West'}
Marketing regions: {'Tier2', 'South', 'East', 'North', 'West'}


In [28]:
## Final cleaning checks

# Primary key validation
primary_keys = {
    "customers": ("customers", "customer_id"),
    "orders": ("orders", "order_id"),
    "customer_features": ("customer_features", "customer_id"),
    "customer_unit_economics": ("customer_unit_economics", "customer_id"),
    "subscription_history": ("subscription_history", "subscription_id")
}

for table, (dataset, column) in primary_keys.items():
    df = datasets[dataset]

    print(
        f"{table}: "
        f"unique={df[column].is_unique}, "
        f"nulls={df[column].isna().sum()}"
    )

customers: unique=True, nulls=0
orders: unique=True, nulls=0
customer_features: unique=True, nulls=0
customer_unit_economics: unique=True, nulls=0
subscription_history: unique=True, nulls=0


In [29]:
## foreign keys check 

customer_ids = set(customers["customer_id"])

foreign_key_tables = [
    "orders",
    "customer_features",
    "customer_unit_economics",
    "subscription_history"
]

for table in foreign_key_tables:
    unmatched = (~datasets[table]["customer_id"].isin(customer_ids)).sum()
    print(f"{table}: {unmatched} unmatched customer IDs")

orders: 0 unmatched customer IDs
customer_features: 0 unmatched customer IDs
customer_unit_economics: 0 unmatched customer IDs
subscription_history: 0 unmatched customer IDs


In [30]:
## Export processed datasets

for name, df in datasets.items():
    output_path = PROCESSED_PATH / f"{name}.csv"

    df.to_csv(output_path, index=False)

    print(f"Saved: {output_path}")

Saved: ../data/processed/customers.csv
Saved: ../data/processed/orders.csv
Saved: ../data/processed/customer_features.csv
Saved: ../data/processed/customer_unit_economics.csv
Saved: ../data/processed/subscription_history.csv
Saved: ../data/processed/marketing_spend.csv
Saved: ../data/processed/date_dim.csv
